In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("Churn_Modelling.csv")
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [2]:
df.shape

(10000, 14)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB


In [4]:
df.duplicated().sum()

np.int64(0)

In [5]:
df["Exited"].value_counts()

Exited
0    7963
1    2037
Name: count, dtype: int64

In [6]:
df["Geography"].value_counts()

Geography
France     5014
Germany    2509
Spain      2477
Name: count, dtype: int64

In [7]:
df["Gender"].value_counts()

Gender
Male      5457
Female    4543
Name: count, dtype: int64

In [8]:
# dropping irrelevant columns implementing feature selection

df.drop(columns = ['RowNumber','CustomerId','Surname'],inplace=True)

In [9]:
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [10]:
df.shape

(10000, 11)

# one hot encoding for categorical variables


In [11]:
df = pd.get_dummies(df,columns=['Geography','Gender'],drop_first=True)
df.head()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Germany,Geography_Spain,Gender_Male
0,619,42,2,0.00,1,1,1,101348.88,1,False,False,False
1,608,41,1,83807.86,1,0,1,112542.58,0,False,True,False
2,502,42,8,159660.80,3,1,0,113931.57,1,False,False,False
3,699,39,1,0.00,2,0,0,93826.63,0,False,False,False
4,850,43,2,125510.82,1,1,1,79084.10,0,False,True,False


In [12]:
df.shape

(10000, 12)

# Train test split and scaling 

In [13]:
X = df.drop(columns=['Exited'])
y = df['Exited'].values

from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=0)

In [14]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train_trf = scaler.fit_transform(X_train)
X_test_trf = scaler.transform(X_test)

In [15]:
X_test_trf.shape

(2000, 11)

# Building the network using tensorflow

In [16]:
import tensorflow
from tensorflow import keras
from tensorflow.keras import Sequential 
from tensorflow.keras.layers import Dense

In [17]:
def create_model():
    model = Sequential([
        Dense(11, activation='relu', input_shape=(X_train_trf.shape[1],)),
        Dense(11, activation='relu'),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer='sgd',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

# Implementing Batch Gradient 

In [18]:
model_batch = create_model()

model_batch.fit(
    X_train_trf,
    y_train,
    batch_size=len(X_train_trf),
    epochs=30,
    verbose=1,
    validation_split=0.2)

C:\Users\Shekhar Kumar\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.2195 - loss: 1.0063 - val_accuracy: 0.2094 - val_loss: 0.9902
Epoch 2/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 405ms/step - accuracy: 0.2219 - loss: 0.9938 - val_accuracy: 0.2125 - val_loss: 0.9783
Epoch 3/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 403ms/step - accuracy: 0.2242 - loss: 0.9818 - val_accuracy: 0.2138 - val_loss: 0.9668
Epoch 4/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 427ms/step - accuracy: 0.2272 - loss: 0.9702 - val_accuracy: 0.2163 - val_loss: 0.9556
Epoch 5/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 283ms/step - accuracy: 0.2291 - loss: 0.9589 - val_accuracy: 0.2206 - val_loss: 0.9449
Epoch 6/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 293ms/step - accuracy: 0.2328 - loss: 0.9480 - val_accuracy: 0.2244 - val_loss: 0.9344
Epoch 7/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 217ms/step - accuracy: 0.2381 - loss: 0.9375 - val_accuracy: 0.2300 - val_loss: 0.9243
Epoch 8/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 205ms/step - accuracy: 0.2427 - loss: 0.9272 - val_accuracy: 0.2313 - val_loss: 0.

In [20]:
model_batch.layers[0].get_weights()

[array([[ 2.1069331e-01, -1.8672849e-01, -3.5502365e-01, -3.4112304e-01,
         -2.2234779e-04,  5.1296633e-01, -4.8539788e-01,  1.0405661e-01,
         -9.2090063e-02, -4.6049720e-03,  3.8777828e-01],
        [ 1.7698398e-01, -4.5271537e-01,  2.1115442e-01,  4.3393049e-01,
          4.3102229e-01,  2.9875976e-01,  4.2616244e-02,  2.8807458e-01,
         -6.1113100e-02,  1.0347843e-01,  4.3630654e-01],
        [ 3.0462611e-01,  1.2105254e-02, -1.4115289e-01,  2.9405200e-01,
          2.7417615e-01, -3.0207375e-01,  5.2220243e-01, -1.2555771e-01,
         -5.1211417e-01,  2.7594054e-01,  6.0312271e-02],
        [ 3.3516252e-01,  3.7648544e-01, -9.5779426e-02,  5.8014430e-03,
          2.3935129e-01, -4.6063063e-01,  3.0646333e-01,  3.5360372e-01,
          4.5158118e-01,  3.6640719e-01, -4.0715179e-01],
        [ 4.4935659e-01, -5.0904029e-03, -3.1300831e-01,  1.9504376e-01,
         -3.0708876e-01,  1.6184294e-01, -3.2438979e-01, -5.1745588e-01,
          9.6763648e-02, -4.5909381e-0

In [21]:
model_batch.layers[1].get_weights()

[array([[ 0.32228237, -0.1652873 ,  0.2750088 , -0.40126818,  0.42663187,
          0.01256921, -0.1825046 ,  0.19744237,  0.14154111, -0.41227812,
          0.4946485 ],
        [ 0.22645591,  0.29408923,  0.03971896,  0.18028203, -0.24321105,
          0.45736134, -0.35216463,  0.48566347, -0.14967772,  0.06355688,
          0.26077506],
        [ 0.20047675,  0.37536603,  0.3298467 , -0.34771413, -0.50666547,
         -0.4506632 ,  0.03999875,  0.00824638, -0.18969508,  0.47717693,
          0.4211952 ],
        [ 0.02709517,  0.14768255,  0.3628322 , -0.46064913,  0.20220108,
          0.12765239, -0.49281263,  0.15473567, -0.25646135, -0.16276227,
          0.20875624],
        [ 0.2865621 , -0.43981642, -0.17125413, -0.24400608, -0.11828103,
          0.16029753,  0.2602117 , -0.32969508,  0.4715825 ,  0.44325018,
         -0.49438822],
        [ 0.4533715 , -0.5103277 ,  0.3876569 , -0.31308612,  0.44956297,
         -0.30374992,  0.06501246, -0.15967849, -0.5068158 , -0.1213743

In [22]:
model_batch.layers[2].get_weights()

[array([[ 0.4974874 ],
        [ 0.13513137],
        [-0.17304677],
        [-0.6264248 ],
        [-0.57222426],
        [-0.48963982],
        [-0.35622704],
        [ 0.3577406 ],
        [ 0.02275775],
        [ 0.25903907],
        [ 0.41008717]], dtype=float32),
 array([-0.10200968], dtype=float32)]

In [19]:
y_log = model_batch.predict(X_test_trf)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step


In [20]:
y_pred = np.where(y_log > 0.5 , 1 , 0)
from sklearn.metrics import accuracy_score
accuracy_score(y_test , y_pred)

0.38

# Implementing Stochastic Gradient Descent

In [21]:
model_sgd = create_model()

model_sgd.fit(
    X_train_trf,
    y_train,
    batch_size=1,
    epochs=3,
    verbose=1,
    validation_split=0.2)

# model_sgd.layers[0].get_weights()

C:\Users\Shekhar Kumar\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/3
6400/6400 ━━━━━━━━━━━━━━━━━━━━ 34s 5ms/step - accuracy: 0.8097 - loss: 0.4461 - val_accuracy: 0.8244 - val_loss: 0.4199
Epoch 2/3
6400/6400 ━━━━━━━━━━━━━━━━━━━━ 32s 5ms/step - accuracy: 0.8258 - loss: 0.4054 - val_accuracy: 0.8194 - val_loss: 0.4059
Epoch 3/3
6400/6400 ━━━━━━━━━━━━━━━━━━━━ 32s 5ms/step - accuracy: 0.8325 - loss: 0.3882 - val_accuracy: 0.8331 - val_loss: 0.3862


In [26]:
model_sgd.layers[1].get_weights()

[array([[-0.14080068, -0.2289193 , -0.11922805,  0.6014932 ,  0.44573563,
         -0.1151396 ,  0.02609706,  0.16900311,  0.25718167,  0.16138741,
          0.3102988 ],
        [-0.45356947, -0.39366162, -0.15624085, -0.2937329 , -0.07287353,
         -0.55494744, -0.0897253 , -0.12600023,  0.3203888 ,  0.09847242,
         -0.424586  ],
        [ 0.12826721,  0.11916425, -0.4631147 ,  0.50481284, -0.06238472,
         -0.16516836,  0.07449918,  0.2028376 ,  0.11753605,  0.15159373,
          0.36838454],
        [ 0.13883437, -0.24482827, -0.31957006, -0.6609127 , -0.21678893,
          0.24990204, -0.21998173, -0.0554263 ,  0.0426238 , -0.08565238,
         -0.07021891],
        [ 0.11103641, -0.43228257,  0.45197332, -0.02417003, -0.41612855,
          0.3670926 , -0.22511709, -0.11193589, -0.23182008,  0.4300746 ,
         -0.12032007],
        [ 0.16995603, -0.11402183, -0.14603418, -0.0647153 , -0.10811961,
         -0.37945327, -0.10293589,  0.10917562, -0.46566218, -0.1952193

In [27]:
model_sgd.layers[2].get_weights()

[array([[ 1.0254037 ],
        [ 0.45685083],
        [-0.35834724],
        [-0.83428895],
        [-0.61272275],
        [ 0.4688326 ],
        [-0.73344487],
        [ 0.51297456],
        [ 0.36986026],
        [-0.3412611 ],
        [-0.652944  ]], dtype=float32),
 array([-0.71250427], dtype=float32)]

In [28]:
y_log = model_sgd.predict(X_test_trf)
y_pred = np.where(y_log > 0.5 , 1 , 0)
accuracy_score(y_test , y_pred)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step  


0.85

# Implementing Mini Batch GD

In [23]:
model_mini = create_model()

model_mini.fit(
    X_train_trf,
    y_train,
    batch_size=64,
    epochs=30,
    verbose=1,
    validation_split=0.2)

Epoch 1/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.7034 - loss: 0.6191 - val_accuracy: 0.7969 - val_loss: 0.5530
Epoch 2/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7953 - loss: 0.5186 - val_accuracy: 0.7969 - val_loss: 0.5071
Epoch 3/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.7952 - loss: 0.4887 - val_accuracy: 0.7944 - val_loss: 0.4838
Epoch 4/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.7977 - loss: 0.4717 - val_accuracy: 0.7994 - val_loss: 0.4696
Epoch 5/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.8017 - loss: 0.4608 - val_accuracy: 0.7956 - val_loss: 0.4605
Epoch 6/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.8053 - loss: 0.4531 - val_accuracy: 0.8012 - val_loss: 0.4543
Epoch 7/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.8075 - loss: 0.4473 - val_accuracy: 0.8037 - val_loss: 0.4498
Epoch 8/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8109 - loss: 0.4425 - val_accuracy: 0

In [30]:
y_log = model_mini.predict(X_test_trf)
y_pred = np.where(y_log > 0.5 , 1 , 0)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step  


In [31]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test , y_pred)

0.799